# Figure 3d: alternative-splicing pattern comparison

This notebook compares alternative-splicing pattern frequencies between
tumor-specific transcripts and non-tumor-specific transcripts across tissues.

The workflow:

1. loads tissue-level transcript–pattern intersection tables;
2. counts transcripts assigned to each splicing pattern;
3. constructs tissue-specific 2 × 2 contingency tables;
4. applies Fisher's exact test;
5. corrects P values using the Benjamini–Hochberg procedure;
6. exports statistical results and the plotting matrix.

All debugging output and exploratory analyses from the original notebook have been
removed.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="white")


## Configuration

Update `project_dir` when running the analysis in another environment.


In [ ]:
project_dir = Path("/data1/HOMO_PANGENOME/DYY/data")
output_dir = project_dir / "bambu" / "figure_3d"
output_dir.mkdir(parents=True, exist_ok=True)

non_tumor_intersection_dir = (
    project_dir / "9.26_venn_intersect_data"
)
tumor_specific_intersection_dir = (
    project_dir / "9.26_venn_intersect_ts_data"
)

significance_cutoff = 0.05
minimum_pattern_length = 3


## Utility functions


In [ ]:
def require_directory(directory):
    """Validate that an input directory exists and contains files."""
    if not directory.is_dir():
        raise FileNotFoundError(f"Directory was not found: {directory}")

    if not any(directory.iterdir()):
        raise FileNotFoundError(f"Directory is empty: {directory}")


def require_columns(table, required_columns, table_name):
    """Validate that a table contains all required columns."""
    missing_columns = sorted(set(required_columns) - set(table.columns))

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: {missing_columns}"
        )


def split_transcript_ids(value):
    """Convert a delimited transcript field into a list of transcript IDs."""
    if pd.isna(value):
        return []

    text = str(value).strip()

    if not text:
        return []

    for delimiter in [",", ";", "|"]:
        if delimiter in text:
            return [
                item.strip()
                for item in text.split(delimiter)
                if item.strip()
            ]

    return [text]


def load_intersection_tables(directory):
    """Load tissue-level transcript–pattern intersection tables."""
    require_directory(directory)

    tables = {}

    for file_path in sorted(directory.glob("*.csv")):
        tissue = file_path.stem[:3]
        table = pd.read_csv(file_path, sep="\t")

        require_columns(
            table,
            {"pattern", "transcript"},
            file_path.name,
        )

        tables[tissue] = table.copy()

    if not tables:
        raise FileNotFoundError(
            f"No tab-delimited CSV files were found in {directory}"
        )

    return tables


def count_pattern_transcripts(intersection_tables):
    """Count unique transcripts assigned to each pattern in every tissue."""
    pattern_counts = {}
    total_transcript_counts = {}

    for tissue, table in intersection_tables.items():
        expanded_rows = []

        for _, row in table[["pattern", "transcript"]].iterrows():
            pattern = str(row["pattern"]).strip()

            for transcript_id in split_transcript_ids(row["transcript"]):
                expanded_rows.append(
                    {
                        "pattern": pattern,
                        "transcript_id": transcript_id,
                    }
                )

        expanded_table = pd.DataFrame(expanded_rows)

        if expanded_table.empty:
            pattern_counts[tissue] = pd.Series(dtype="int64")
            total_transcript_counts[tissue] = 0
            continue

        expanded_table = expanded_table.drop_duplicates(
            subset=["pattern", "transcript_id"]
        )

        pattern_counts[tissue] = (
            expanded_table.groupby("pattern")["transcript_id"]
            .nunique()
        )
        total_transcript_counts[tissue] = (
            expanded_table["transcript_id"].nunique()
        )

    count_table = (
        pd.DataFrame(pattern_counts)
        .fillna(0)
        .astype(int)
    )

    total_counts = pd.Series(
        total_transcript_counts,
        name="total_transcripts",
        dtype="int64",
    )

    return count_table, total_counts


def calculate_pattern_statistics(
    non_tumor_counts,
    tumor_counts,
    non_tumor_totals,
    tumor_totals,
):
    """Compare pattern frequencies using tissue-specific Fisher tests."""
    tissues = sorted(
        set(non_tumor_counts.columns)
        & set(tumor_counts.columns)
        & set(non_tumor_totals.index)
        & set(tumor_totals.index)
    )

    patterns = sorted(
        set(non_tumor_counts.index)
        | set(tumor_counts.index)
    )

    result_rows = []

    for tissue in tissues:
        non_tumor_total = int(non_tumor_totals.loc[tissue])
        tumor_total = int(tumor_totals.loc[tissue])

        for pattern in patterns:
            non_tumor_pattern = int(
                non_tumor_counts.get(tissue, pd.Series(dtype=int))
                .get(pattern, 0)
            )
            tumor_pattern = int(
                tumor_counts.get(tissue, pd.Series(dtype=int))
                .get(pattern, 0)
            )

            non_tumor_other = non_tumor_total - non_tumor_pattern
            tumor_other = tumor_total - tumor_pattern

            if min(
                non_tumor_pattern,
                tumor_pattern,
                non_tumor_other,
                tumor_other,
            ) < 0:
                raise ValueError(
                    f"Invalid contingency table for {tissue}, {pattern}"
                )

            odds_ratio, p_value = fisher_exact(
                [
                    [tumor_pattern, tumor_other],
                    [non_tumor_pattern, non_tumor_other],
                ],
                alternative="two-sided",
            )

            tumor_proportion = (
                tumor_pattern / tumor_total
                if tumor_total > 0
                else np.nan
            )
            non_tumor_proportion = (
                non_tumor_pattern / non_tumor_total
                if non_tumor_total > 0
                else np.nan
            )

            result_rows.append(
                {
                    "tissue": tissue,
                    "pattern": pattern,
                    "tumor_specific_count": tumor_pattern,
                    "tumor_specific_total": tumor_total,
                    "non_tumor_count": non_tumor_pattern,
                    "non_tumor_total": non_tumor_total,
                    "tumor_specific_proportion": tumor_proportion,
                    "non_tumor_proportion": non_tumor_proportion,
                    "proportion_difference": (
                        tumor_proportion - non_tumor_proportion
                    ),
                    "odds_ratio": odds_ratio,
                    "p_value": p_value,
                }
            )

    results = pd.DataFrame(result_rows)

    if results.empty:
        return results

    results["adjusted_p_value"] = multipletests(
        results["p_value"],
        method="fdr_bh",
    )[1]

    results["significant"] = (
        results["adjusted_p_value"] < significance_cutoff
    )

    return results


## Load tissue-level intersection data


In [ ]:
non_tumor_tables = load_intersection_tables(
    non_tumor_intersection_dir
)
tumor_specific_tables = load_intersection_tables(
    tumor_specific_intersection_dir
)

shared_tissues = sorted(
    set(non_tumor_tables) & set(tumor_specific_tables)
)

if not shared_tissues:
    raise ValueError(
        "No shared tissues were found between the two input directories."
    )

non_tumor_tables = {
    tissue: non_tumor_tables[tissue]
    for tissue in shared_tissues
}
tumor_specific_tables = {
    tissue: tumor_specific_tables[tissue]
    for tissue in shared_tissues
}


## Count transcripts by splicing pattern


In [ ]:
non_tumor_pattern_counts, non_tumor_totals = (
    count_pattern_transcripts(non_tumor_tables)
)

tumor_specific_pattern_counts, tumor_specific_totals = (
    count_pattern_transcripts(tumor_specific_tables)
)

all_patterns = sorted(
    set(non_tumor_pattern_counts.index)
    | set(tumor_specific_pattern_counts.index)
)

selected_patterns = [
    pattern
    for pattern in all_patterns
    if len(str(pattern)) >= minimum_pattern_length
]


## Statistical comparison

A two-sided Fisher's exact test is performed for every tissue–pattern combination.
P values are adjusted across all tests using the Benjamini–Hochberg method.


In [ ]:
pattern_statistics = calculate_pattern_statistics(
    non_tumor_counts=non_tumor_pattern_counts,
    tumor_counts=tumor_specific_pattern_counts,
    non_tumor_totals=non_tumor_totals,
    tumor_totals=tumor_specific_totals,
)

pattern_statistics = pattern_statistics.loc[
    pattern_statistics["pattern"].isin(selected_patterns)
].copy()

pattern_statistics.to_csv(
    output_dir / "figure_3d_pattern_statistics.tsv",
    sep="\t",
    index=False,
)


## Prepare the Figure 3d plotting matrix


In [ ]:
plot_matrix = pattern_statistics.pivot(
    index="pattern",
    columns="tissue",
    values="proportion_difference",
)

significance_matrix = pattern_statistics.pivot(
    index="pattern",
    columns="tissue",
    values="adjusted_p_value",
)

plot_matrix = plot_matrix.loc[
    plot_matrix.abs().max(axis=1).sort_values(ascending=False).index
]

significance_matrix = significance_matrix.reindex(
    index=plot_matrix.index,
    columns=plot_matrix.columns,
)

plot_matrix.to_csv(
    output_dir / "figure_3d_proportion_difference.tsv",
    sep="\t",
)

significance_matrix.to_csv(
    output_dir / "figure_3d_adjusted_pvalues.tsv",
    sep="\t",
)


## Plot Figure 3d

Positive values indicate enrichment among tumor-specific transcripts; negative values
indicate enrichment among non-tumor-specific transcripts. Asterisks denote
Benjamini–Hochberg adjusted P values below 0.05.


In [ ]:
if plot_matrix.empty:
    raise ValueError("No splicing patterns remained after filtering.")

figure_width = max(6, 0.55 * plot_matrix.shape[1] + 3)
figure_height = max(4, 0.28 * plot_matrix.shape[0] + 2)

figure, axis = plt.subplots(
    figsize=(figure_width, figure_height)
)

sns.heatmap(
    plot_matrix,
    cmap="vlag",
    center=0,
    linewidths=0.4,
    linecolor="white",
    cbar_kws={
        "label": (
            "Tumor-specific proportion − "
            "non-tumor-specific proportion"
        )
    },
    ax=axis,
)

for row_index, pattern in enumerate(plot_matrix.index):
    for column_index, tissue in enumerate(plot_matrix.columns):
        adjusted_p_value = significance_matrix.loc[
            pattern,
            tissue,
        ]

        if (
            pd.notna(adjusted_p_value)
            and adjusted_p_value < significance_cutoff
        ):
            axis.text(
                column_index + 0.5,
                row_index + 0.5,
                "*",
                ha="center",
                va="center",
            )

axis.set_xlabel("Tissue")
axis.set_ylabel("Alternative-splicing pattern")
axis.set_title(
    "Alternative-splicing pattern enrichment"
)

figure.savefig(
    output_dir / "figure_3d_splicing_pattern_heatmap.pdf",
    bbox_inches="tight",
)

figure.savefig(
    output_dir / "figure_3d_splicing_pattern_heatmap.png",
    dpi=400,
    bbox_inches="tight",
)

plt.show()


## Reproducibility notes

- All execution counters and cell outputs were cleared before release.
- Temporary gene lookups, manual pattern lists, duplicated statistical tests, and
  unrelated exploratory plots were removed.
- Transcript counts are based on unique transcript identifiers.
- Tissue matching and pattern alignment are performed explicitly.
- Statistical testing uses the raw integer counts rather than rounded proportions.
